# Day 25 – Clustering & Unsupervised Learning
## Finding Natural Groups When There Is No Target

Unsupervised learning discovers structure in data without a predefined target.

**Framework:** Prepare → Explore → Cluster → Evaluate → Interpret → Act


## 1. Create a Synthetic Ward-Level Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(25)
n=120
df=pd.DataFrame({
    "ward":[f"Ward_{i:03d}" for i in range(1,n+1)],
    "complaint_volume":np.random.randint(400,5000,n),
    "backlog":np.random.randint(30,900,n),
    "avg_resolution_days":np.random.uniform(2,18,n),
    "staff_available":np.random.randint(20,100,n),
    "repeat_complaint_rate":np.random.uniform(.05,.40,n),
    "priority_share":np.random.uniform(.05,.35,n)
})
df["complaints_per_staff"]=df["complaint_volume"]/df["staff_available"]
df["backlog_per_staff"]=df["backlog"]/df["staff_available"]
df.head()


## 2. Select and Scale Clustering Features

In [ ]:
from sklearn.preprocessing import StandardScaler

features=["complaint_volume","backlog","avg_resolution_days",
           "staff_available","repeat_complaint_rate","priority_share"]
X=df[features]
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)


## 3. K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans

kmeans=KMeans(n_clusters=4,random_state=25,n_init=10)
df["cluster"]=kmeans.fit_predict(X_scaled)
df["cluster"].value_counts().sort_index()


## 4. Profile the Clusters

In [ ]:
cluster_profile=df.groupby("cluster")[features].mean().round(2)
cluster_profile


## 5. Visualize the Clusters

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df["complaint_volume"],df["avg_resolution_days"],
            c=df["cluster"],alpha=.75)
plt.xlabel("Complaint Volume")
plt.ylabel("Average Resolution Days")
plt.title("Ward Clusters: Workload vs Resolution Time")
plt.tight_layout()
plt.show()


## 6. Elbow Method

In [ ]:
inertias=[]
for k in range(2,11):
    model=KMeans(n_clusters=k,random_state=25,n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2,11),inertias,marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("K-Means Elbow Analysis")
plt.tight_layout()
plt.show()


## 7. Silhouette Score

In [ ]:
from sklearn.metrics import silhouette_score

scores=[]
for k in range(2,11):
    model=KMeans(n_clusters=k,random_state=25,n_init=10)
    labels=model.fit_predict(X_scaled)
    scores.append(silhouette_score(X_scaled,labels))

pd.DataFrame({"K":range(2,11),"Silhouette Score":scores}).round(3)


## 8. Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import linkage,dendrogram

Z=linkage(X_scaled[:40],method="ward")
plt.figure(figsize=(10,5))
dendrogram(Z,no_labels=True)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Ward observations")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()


## 9. PCA for Cluster Visualization

In [ ]:
from sklearn.decomposition import PCA

pca=PCA(n_components=2)
X_2d=pca.fit_transform(X_scaled)

plt.figure(figsize=(8,5))
plt.scatter(X_2d[:,0],X_2d[:,1],c=df["cluster"],alpha=.75)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters in PCA Space")
plt.tight_layout()
plt.show()

print("Variance explained:",round(pca.explained_variance_ratio_.sum(),3))


## 10. Interpret the Clusters

Cluster numbers have no inherent meaning. Interpret them through their profiles.

Possible operational segments include:
- High workload / high backlog
- Lower workload / faster resolution
- High repeat-complaint areas
- High service-delay areas

These labels are examples only. The actual interpretation must come from the observed cluster profile.

## 11. Important Caveats

- Scaling and feature choice affect clustering.
- K-Means is distance-based and sensitive to outliers.
- Different K values can produce different structures.
- Cluster labels are arbitrary.
- Statistical separation does not guarantee operational usefulness.
- Check stability over time before using clusters for decisions.

## Key Takeaway

**Clustering discovers structure without a predefined target.**

For public-sector analytics, it can support segmentation of wards, complaint patterns, service areas and operational workloads.

A cluster becomes valuable when it leads to a meaningful operational interpretation or decision.

**Next:** Day 26 – Time Series Analysis & Forecasting.
